# CENG428 Neural Networks — Synthetic Shape Detection

**Framework:** PyTorch + torchvision  
**Base dataset:** MS COCO 2017  
**Task:** Detect or segment synthetic shapes added to natural images

In [ ]:
import hashlib
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFilter
from torchvision.datasets import CocoDetection
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Create the project directories used during setup, training, and evaluation.
for path in [
    Path("data"),
    Path("results"),
    Path("figures"),
    Path("figures/generated_samples"),
    Path("figures/prediction_visualizations"),
]:
    path.mkdir(parents=True, exist_ok=True)

print("Directory setup completed.")

---
## 2. Dataset and Split

In [ ]:
# Assignment constants
GLOBAL_SEED          = 2025
TRAIN_SIZE           = 5000   # first 5000 images from train2017
VAL_SIZE             = 1000   # first 1000 images from val2017
TEST_SIZE            = 1000   # next  1000 images from val2017 (indices 1000–1999)
POSITIVE_RATIO       = 0.70   # 70% positive, 30% negative
MAX_SHAPES_PER_IMAGE = 3      # 1 to 3 shapes per positive image


# Deterministic seed
# Do NOT use Python's built-in hash(); output varies between sessions.
def make_seed(split_name: str, image_id: int, global_seed: int = GLOBAL_SEED) -> int:
    key = f"{split_name}_{image_id}_{global_seed}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:8], 16)


# COCO base initialization
# Required directory structure:
#   data/coco/
#   ├── train2017/
#   ├── val2017/
#   └── annotations/
#       ├── instances_train2017.json
#       └── instances_val2017.json
def build_coco_bases(data_root: str | Path = "data/coco"):
    root = Path(data_root)

    train_base = CocoDetection(
        root=str(root / "train2017"),
        annFile=str(root / "annotations" / "instances_train2017.json"),
    )

    val_base = CocoDetection(
        root=str(root / "val2017"),
        annFile=str(root / "annotations" / "instances_val2017.json"),
    )

    return train_base, val_base


# Fixed split protocol
#   - Sort all image IDs in increasing order
#   - train : first 5000 from train2017
#   - val   : first 1000 from val2017
#   - test  : next  1000 from val2017  (do NOT use for model selection)
def get_split_ids(
    coco_base: CocoDetection,
    split: Literal["train", "val", "test"],
) -> list[int]:
    all_ids = sorted(coco_base.coco.getImgIds())

    if split == "train":
        return all_ids[:TRAIN_SIZE]
    elif split == "val":
        return all_ids[:VAL_SIZE]
    elif split == "test":
        return all_ids[VAL_SIZE : VAL_SIZE + TEST_SIZE]
    else:
        raise ValueError(f"Unknown split: {split!r}")

In [ ]:
train_base, val_base = build_coco_bases("data/coco")

train_ids = get_split_ids(train_base, "train")   # first 5000 from train2017
val_ids   = get_split_ids(val_base,   "val")     # first 1000 from val2017
test_ids  = get_split_ids(val_base,   "test")    # next  1000 from val2017

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

---
## 3. Synthetic Shape Generation

Synthetic samples are created by drawing random shapes directly onto COCO background images. The current generator supports rectangles, ellipses, and triangles, and varies location, size, color, opacity, and the number of shapes per image.

Positive images contain between **1 and 3** synthetic shapes, while negative images contain no target shape; the positive/negative ratio is approximately **70/30** as suggested in the assignment.

To keep the task nontrivial, the generator already includes multiple difficulty mechanisms such as random opacity, low-contrast colors sampled from local image content, Gaussian blur on inserted shapes, and additive noise on the final image.

Part of the synthetic shape placement and bounding-box design was inspired by prior experience from a Teknofest Aviation AI project. However, for this assignment, all synthetic shapes and their corresponding labels are generated fully automatically, and no manual annotation is used.

In [ ]:
@dataclass
class ShapeGenerationConfig:
    opacity_range: tuple[int, int] = (96, 220)
    blur_range: tuple[float, float] = (0.2, 1.6)
    noise_std_range: tuple[float, float] = (2.0, 8.0)
    color_jitter: int = 35
    min_size_ratio: float = 1 / 12
    max_size_ratio: float = 1 / 4
    opaque_probability: float = 0.6
    opaque_alpha_range: tuple[int, int] = (220, 255)
    transparent_alpha_range: tuple[int, int] = (80, 180)


DEFAULT_SHAPE_CONFIG = ShapeGenerationConfig()


class ShapeGenerator:
    def __init__(self, config: ShapeGenerationConfig | None = None):
        self.config = config or DEFAULT_SHAPE_CONFIG

    def _sample_alpha(self, rng: random.Random) -> int:
        if rng.random() < self.config.opaque_probability:
            return rng.randint(*self.config.opaque_alpha_range)
        return rng.randint(*self.config.transparent_alpha_range)

    def _sample_color(self, image_np: np.ndarray, x1: int, y1: int, x2: int, y2: int, rng: random.Random):
        patch = image_np[y1:y2, x1:x2]
        if patch.size == 0:
            patch = image_np

        base = patch.reshape(-1, 3).mean(axis=0)
        jitter = np.array([
            rng.randint(-self.config.color_jitter, self.config.color_jitter) for _ in range(3)
        ], dtype=np.float32)
        color = np.clip(base + jitter, 0, 255).astype(np.uint8)
        alpha = self._sample_alpha(rng)
        return tuple(int(v) for v in color), alpha

    def _draw_shape(self, draw, shape_type: str, box, fill, rng: random.Random, for_mask: bool = False):
        x1, y1, x2, y2 = box
        if shape_type == "rectangle":
            draw.rectangle(box, fill=fill)
        elif shape_type == "ellipse":
            draw.ellipse(box, fill=fill)
        elif shape_type == "circle":
            side = min(x2 - x1, y2 - y1)
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            draw.ellipse((cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2), fill=fill)
        elif shape_type == "triangle":
            pts = [((x1 + x2) / 2, y1), (x1, y2), (x2, y2)]
            draw.polygon(pts, fill=fill)
        elif shape_type == "line":
            width = max(2, int(min(x2 - x1, y2 - y1) * 0.12))
            width = max(width, 1 if for_mask else width)
            draw.line((x1, y2, x2, y1), fill=fill, width=width)
        else:
            raise ValueError(f"Unsupported shape type: {shape_type}")

    def generate(self, image, n_shapes: int, rng: random.Random):
        """
        Draw n_shapes synthetic shapes onto image.

        Args:
            image    : PIL.Image (RGB)
            n_shapes : number of shapes to draw (0 for negative images)
            rng      : seeded random.Random instance

        Returns:
            augmented_image : PIL.Image
            boxes           : list[list[float]]   [[x1, y1, x2, y2], ...]
            mask            : np.ndarray [H, W]   1=shape pixel, 0=background
        """
        image = image.convert("RGB")
        image_np = np.array(image)
        h, w = image_np.shape[:2]

        canvas = image.convert("RGBA")
        binary_mask = np.zeros((h, w), dtype=np.uint8)
        boxes = []
        shape_types = ["rectangle", "ellipse", "triangle", "circle", "line"]

        min_w = max(12, int(w * self.config.min_size_ratio))
        max_w = max(min_w + 1, int(w * self.config.max_size_ratio))
        min_h = max(12, int(h * self.config.min_size_ratio))
        max_h = max(min_h + 1, int(h * self.config.max_size_ratio))

        for _ in range(n_shapes):
            box_w = rng.randint(min_w, max_w)
            box_h = rng.randint(min_h, max_h)
            shape_type = rng.choice(shape_types)

            if shape_type == "circle":
                side = min(box_w, box_h)
                box_w = side
                box_h = side

            x1 = rng.randint(0, max(0, w - box_w - 1))
            y1 = rng.randint(0, max(0, h - box_h - 1))
            x2 = min(w - 1, x1 + box_w)
            y2 = min(h - 1, y1 + box_h)

            color_rgb, alpha = self._sample_color(image_np, x1, y1, x2, y2, rng)
            fill_rgba = (*color_rgb, alpha)
            box = (x1, y1, x2, y2)
            center = ((x1 + x2) / 2, (y1 + y2) / 2)
            rotation_angle = rng.uniform(0, 180) if shape_type in {"rectangle", "triangle", "line"} else 0.0

            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            overlay_draw = ImageDraw.Draw(overlay)
            self._draw_shape(overlay_draw, shape_type, box, fill_rgba, rng=rng, for_mask=False)

            shape_mask = Image.new("L", (w, h), 0)
            mask_draw = ImageDraw.Draw(shape_mask)
            self._draw_shape(mask_draw, shape_type, box, 1, rng=rng, for_mask=True)

            if rotation_angle:
                overlay = overlay.rotate(rotation_angle, resample=Image.Resampling.BICUBIC, center=center)
                shape_mask = shape_mask.rotate(rotation_angle, resample=Image.Resampling.NEAREST, center=center)

            blur_radius = rng.uniform(*self.config.blur_range)
            overlay = overlay.filter(ImageFilter.GaussianBlur(radius=blur_radius))
            canvas = Image.alpha_composite(canvas, overlay)

            shape_mask_np = np.array(shape_mask, dtype=np.uint8)
            if shape_mask_np.max() == 0:
                continue
            ys, xs = np.where(shape_mask_np > 0)
            boxes.append([float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())])
            binary_mask = np.maximum(binary_mask, shape_mask_np)

        augmented = np.array(canvas.convert("RGB"), dtype=np.float32)
        noise_std = rng.uniform(*self.config.noise_std_range)
        np_rng = np.random.default_rng(rng.randint(0, 2**32 - 1))
        augmented = np.clip(augmented + np_rng.normal(0.0, noise_std, augmented.shape), 0, 255).astype(np.uint8)
        augmented_image = Image.fromarray(augmented)

        return augmented_image, boxes, binary_mask


In [ ]:
# Visualize at least 12 generated examples (§4).
# Must include both positive and negative samples.

def show_generated_samples(n_positive: int = 6, n_negative: int = 6, save_dir: str | Path = "figures/generated_samples"):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    generator = ShapeGenerator(DEFAULT_SHAPE_CONFIG)
    examples = []

    for idx in range(n_positive):
        image_id = val_ids[idx]
        coco_idx = val_base.ids.index(image_id)
        image, _ = val_base[coco_idx]
        rng = random.Random(make_seed("val", image_id))
        n_shapes = rng.randint(1, MAX_SHAPES_PER_IMAGE)
        augmented, boxes, _ = generator.generate(image, n_shapes, rng)
        examples.append((augmented, boxes, f"positive | shapes={len(boxes)}"))

    for idx in range(n_negative):
        image_id = val_ids[n_positive + idx]
        coco_idx = val_base.ids.index(image_id)
        image, _ = val_base[coco_idx]
        rng = random.Random(make_seed("val", image_id))
        augmented, boxes, _ = generator.generate(image, 0, rng)
        examples.append((augmented, boxes, "negative | shapes=0"))

    rows = 3
    cols = 4
    fig, axes = plt.subplots(rows, cols, figsize=(16, 12))
    axes = axes.flatten()

    for ax, (image, boxes, title) in zip(axes, examples):
        ax.imshow(image)
        for x1, y1, x2, y2 in boxes:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(examples):]:
        ax.axis("off")

    fig.tight_layout()
    save_path = save_dir / "generated_samples_grid.png"
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved generated sample grid to {save_path}")


show_generated_samples()

---
## 4. Label Generation

Choose **Option A** (Object Detection) or **Option B** (Semantic Segmentation).

A classification-only model is **not sufficient** for full credit (§5).

In [ ]:
# PyTorch Dataset wrapper
#
# The wrapper must (§2):
#   1. Load a COCO image
#   2. Add one or more synthetic shapes
#   3. Generate the corresponding target label automatically
#   4. Return the modified image and generated target
#
# Target format — Option A: Object Detection (§5)
#   target = {
#       "boxes":       FloatTensor[N, 4]   # x_min, y_min, x_max, y_max
#       "labels":      LongTensor[N]        # all synthetic shapes share label 1
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: boxes = zeros((0,4)), labels = zeros((0,))
#
# Target format — Option B: Semantic Segmentation (§5)
#   target = {
#       "mask":        LongTensor or FloatTensor[H, W]   # 1=shape, 0=background
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: mask contains only zeros
class SyntheticShapeDataset(Dataset):
    def __init__(
        self,
        coco_base: CocoDetection,
        image_ids: list[int],
        split: Literal["train", "val", "test"],
        task: Literal["detection", "segmentation"],
        transform=None,
        generator_config: ShapeGenerationConfig | None = None,
        positive_ratio: float = POSITIVE_RATIO,
        max_shapes_per_image: int = MAX_SHAPES_PER_IMAGE,
    ):
        self.coco_base = coco_base
        self.image_ids = image_ids
        self.split     = split
        self.task      = task
        self.transform = transform
        self.generator = ShapeGenerator(generator_config)
        self.positive_ratio = positive_ratio
        self.max_shapes_per_image = max_shapes_per_image
        self.id_to_index = {img_id: i for i, img_id in enumerate(self.coco_base.ids)}

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        image_id = self.image_ids[idx]

        # Seed: random for train, deterministic for val/test (§3)
        if self.split == "train":
            rng = random.Random()
        else:
            rng = random.Random(make_seed(self.split, image_id))

        # Positive / negative decision — 70% positive, 30% negative (§4)
        is_positive = rng.random() < self.positive_ratio
        n_shapes    = rng.randint(1, self.max_shapes_per_image) if is_positive else 0

        coco_idx = self.id_to_index[image_id]
        image, _ = self.coco_base[coco_idx]
        augmented_image, boxes, mask = self.generator.generate(image, n_shapes, rng)

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        if len(boxes) == 0:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.long)
        else:
            labels_tensor = torch.ones((len(boxes),), dtype=torch.long)

        if self.transform is not None:
            augmented_image = self.transform(augmented_image)
        else:
            augmented_image = T.ToTensor()(augmented_image)

        if self.task == "detection":
            target = {
                "boxes": boxes_tensor,
                "labels": labels_tensor,
                "image_id": image_id,
                "is_positive": is_positive,
            }
        else:
            target = {
                "mask": torch.as_tensor(mask, dtype=torch.long),
                "image_id": image_id,
                "is_positive": is_positive,
            }

        return augmented_image, target

---
## 5. Model Architecture

Use a CNN-based architecture (§6). May train from scratch or fine-tune a pretrained model.
Clearly state which parts are pretrained and which are trained by you.

Detection options: Faster R-CNN, SSD-style, YOLO-style, custom CNN detector.  
Segmentation options: U-Net, FCN-style, encoder-decoder CNN, custom CNN.

In [ ]:
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def build_model(task: str, pretrained: bool):
    """
    Build and return the main detection model.
    For this notebook we implement Faster R-CNN with a ResNet-50 FPN v2 backbone.
    """
    if task != "detection":
        raise NotImplementedError("Only detection is implemented in this notebook.")

    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = fasterrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)

    num_classes = 2  # background + synthetic_shape
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


TASK      = "detection"   # or "segmentation"
PRETRAINED = True
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(TASK, PRETRAINED).to(DEVICE)

---
## 6. Training Procedure

Required training details to report (§7):

| Detail | Value |
|--------|-------|
| Input image size | |
| Batch size | |
| Number of epochs | |
| Optimizer | |
| Learning rate | |
| Loss function | |
| Pretrained weights | |
| Hardware | |
| Approximate training time | |

In [ ]:
import torchvision.transforms as T

def collate_fn(batch):
    return tuple(zip(*batch))


def get_transforms(split: str, use_augmentation: bool = False):
    """Return simple detection-safe transforms.
    We keep transforms light here to avoid changing bounding boxes.
    """
    transforms = []
    if split == "train" and use_augmentation:
        transforms.extend([
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
        ])
    transforms.append(T.ToTensor())
    return T.Compose(transforms)


train_ds = SyntheticShapeDataset(train_base, train_ids, "train", TASK, get_transforms("train"))
val_ds   = SyntheticShapeDataset(val_base,   val_ids,   "val",   TASK, get_transforms("val"))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


def _move_targets_to_device(targets, device):
    moved = []
    for target in targets:
        moved.append({
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in target.items()
        })
    return moved


def train_one_epoch(model, loader, optimizer, device) -> float:
    """Run one training epoch; return average loss."""
    model.train()
    total_loss = 0.0
    num_batches = 0

    for images, targets in loader:
        images = [image.to(device) for image in images]
        targets = _move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_value for loss_value in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        num_batches += 1

    return total_loss / max(num_batches, 1)


def evaluate_val(model, loader, device) -> dict:
    """Evaluate on validation set; return metric dict. Do NOT use on test set for model selection."""
    model.eval()
    predictions, targets_all = [], []

    with torch.no_grad():
        for images, targets in loader:
            images_device = [image.to(device) for image in images]
            outputs = model(images_device)

            outputs_cpu = [
                {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in output.items()}
                for output in outputs
            ]
            targets_cpu = [
                {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in target.items()}
                for target in targets
            ]

            predictions.extend(outputs_cpu)
            targets_all.extend(targets_cpu)

    return detection_metrics(predictions, targets_all)


import time, json
history = []

for epoch in range(1, 11):
    t0 = time.time()
    train_loss  = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_val(model, val_loader, DEVICE)
    elapsed     = time.time() - t0

    print(f"Epoch {epoch:02d}  loss={train_loss:.4f}  {val_metrics}  ({elapsed:.1f}s)")
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

Path("results").mkdir(exist_ok=True)
Path("results/metrics.json").write_text(json.dumps(history, indent=2))

---
## 7. Baseline Method

Compare the CNN model against at least one simple baseline (§8).
Options: color thresholding, edge detection + connected components,
template matching, logistic regression on handcrafted features, or shallow CNN.

In [ ]:
import cv2


class Baseline:
    """
    Simple baseline to compare against the CNN model (§8).
    Choose one of the options listed in §8.
    """
    def __init__(self, canny_low: int = 60, canny_high: int = 160, min_area: int = 64):
        self.canny_low = canny_low
        self.canny_high = canny_high
        self.min_area = min_area

    def _to_numpy_image(self, image):
        if isinstance(image, torch.Tensor):
            image = image.detach().cpu().permute(1, 2, 0).numpy()
            image = np.clip(image * 255.0, 0, 255).astype(np.uint8)
            return image

        if isinstance(image, Image.Image):
            return np.array(image.convert("RGB"))

        return np.asarray(image)

    def predict(self, image):
        image_np = self._to_numpy_image(image)
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, self.canny_low, self.canny_high)

        kernel = np.ones((3, 3), dtype=np.uint8)
        merged = cv2.dilate(edges, kernel, iterations=1)
        num_labels, _, stats, _ = cv2.connectedComponentsWithStats((merged > 0).astype(np.uint8), connectivity=8)

        boxes = []
        scores = []
        image_area = float(image_np.shape[0] * image_np.shape[1])

        for label_idx in range(1, num_labels):
            x, y, w, h, area = stats[label_idx]
            if area < self.min_area:
                continue
            boxes.append([float(x), float(y), float(x + w), float(y + h)])
            scores.append(min(0.99, 0.05 + float(area) / image_area))

        if boxes:
            boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
            scores_tensor = torch.tensor(scores, dtype=torch.float32)
            labels_tensor = torch.ones((len(boxes),), dtype=torch.long)
        else:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            scores_tensor = torch.zeros((0,), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.long)

        return {
            "boxes": boxes_tensor,
            "scores": scores_tensor,
            "labels": labels_tensor,
        }

---
## 8. Evaluation Metrics

In [ ]:
# Fixed test split
# Do NOT use for model selection (§7)
test_ds     = SyntheticShapeDataset(val_base, test_ids, "test", TASK, get_transforms("val"))
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)


def _to_box_tensor(boxes) -> torch.Tensor:
    if isinstance(boxes, torch.Tensor):
        return boxes.detach().cpu().to(torch.float32)
    if boxes is None:
        return torch.zeros((0, 4), dtype=torch.float32)
    return torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def _box_iou_matrix(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)

    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (bottom_right - top_left).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]

    area1 = ((boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0))
    area2 = ((boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0))
    union = area1[:, None] + area2[None, :] - inter

    return torch.where(union > 0, inter / union, torch.zeros_like(inter))


def _match_boxes_greedy(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor, iou_threshold: float):
    ious = _box_iou_matrix(pred_boxes, gt_boxes)
    matches = []

    while ious.numel() > 0:
        max_iou = float(ious.max().item())
        if max_iou < iou_threshold:
            break

        flat_idx = int(ious.argmax().item())
        pred_idx = flat_idx // ious.shape[1]
        gt_idx = flat_idx % ious.shape[1]
        matches.append((pred_idx, gt_idx, max_iou))
        ious[pred_idx, :] = -1.0
        ious[:, gt_idx] = -1.0

    return matches


def detection_metrics(predictions: list, targets: list, iou_threshold: float = 0.5) -> dict:
    """
    Required (§9): Precision@0.5, Recall@0.5, F1@0.5, mean IoU of matched predictions.
    Match predicted boxes to ground-truth boxes via greedy IoU matching.
    """
    tp = 0
    fp = 0
    fn = 0
    matched_ious = []

    for pred, tgt in zip(predictions, targets):
        pred_boxes = _to_box_tensor(pred.get("boxes"))
        gt_boxes = _to_box_tensor(tgt.get("boxes"))

        if "scores" in pred and len(pred["scores"]) == len(pred_boxes):
            order = torch.argsort(pred["scores"].detach().cpu(), descending=True)
            pred_boxes = pred_boxes[order]

        matches = _match_boxes_greedy(pred_boxes, gt_boxes, iou_threshold)
        tp += len(matches)
        fp += len(pred_boxes) - len(matches)
        fn += len(gt_boxes) - len(matches)
        matched_ious.extend(match_iou for _, _, match_iou in matches)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    mean_iou = float(sum(matched_ious) / len(matched_ious)) if matched_ious else 0.0

    return {
        "precision@0.5": precision,
        "recall@0.5": recall,
        "f1@0.5": f1,
        "mean_iou_matched": mean_iou,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
    }


def segmentation_metrics(predictions: list, targets: list) -> dict:
    """
    Required (§9): foreground IoU, Dice coefficient, foreground precision, foreground recall.
    Pixel accuracy alone is NOT sufficient.
    """
    raise NotImplementedError


def evaluate_model_on_loader(model, loader, device) -> dict:
    model.eval()
    predictions, targets_all, images_all = [], [], []

    with torch.no_grad():
        for images, targets in loader:
            outputs = model([image.to(device) for image in images])

            outputs_cpu = [
                {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in output.items()}
                for output in outputs
            ]
            targets_cpu = [
                {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in target.items()}
                for target in targets
            ]

            predictions.extend(outputs_cpu)
            targets_all.extend(targets_cpu)
            images_all.extend([image.detach().cpu() for image in images])

    metrics = detection_metrics(predictions, targets_all)
    return {
        "metrics": metrics,
        "predictions": predictions,
        "targets": targets_all,
        "images": images_all,
    }


def evaluate_baseline_on_loader(baseline, loader) -> dict:
    predictions, targets_all, images_all = [], [], []

    for images, targets in loader:
        batch_predictions = [baseline.predict(image) for image in images]
        targets_cpu = [
            {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in target.items()}
            for target in targets
        ]

        predictions.extend(batch_predictions)
        targets_all.extend(targets_cpu)
        images_all.extend([image.detach().cpu() if isinstance(image, torch.Tensor) else image for image in images])

    metrics = detection_metrics(predictions, targets_all)
    return {
        "metrics": metrics,
        "predictions": predictions,
        "targets": targets_all,
        "images": images_all,
    }


model.eval()
all_predictions, all_targets, all_images = [], [], []

main_model_results = evaluate_model_on_loader(model, test_loader, DEVICE)
test_metrics = main_model_results["metrics"]
all_predictions = main_model_results["predictions"]
all_targets = main_model_results["targets"]
all_images = main_model_results["images"]

baseline = Baseline()
baseline_results = evaluate_baseline_on_loader(baseline, test_loader)
baseline_test_metrics = baseline_results["metrics"]

print("Main model test metrics:", test_metrics)
print("Baseline test metrics:", baseline_test_metrics)

---
## 9. Experiments and Results

At least **3** meaningful experiments required (§10).  
For each: report quantitative results **and** a short interpretation (do not only list numbers).

Planned experiments for later runs:

1. **High opacity vs low opacity**  
   Compare two opacity ranges while keeping the model and data split fixed.
2. **Easy shapes vs hard shapes**  
   Compare simpler synthetic overlays against harder low-contrast / noisy / blurred overlays.
3. **Small training set vs larger training set**  
   Example: first 1000 training images vs first 5000 training images.
4. **With data augmentation vs without data augmentation**  
   Keep the same model and dataset, and only change the training transforms.


In [ ]:
import csv
from IPython.display import Markdown, display


def get_prediction_save_dir(experiment_group: str, variant: str) -> Path:
    """Return a structured directory for prediction visualizations."""
    save_dir = Path("figures") / "prediction_visualizations" / experiment_group / variant
    save_dir.mkdir(parents=True, exist_ok=True)
    return save_dir


def _format_metric_value(value):
    if isinstance(value, (float, np.floating)):
        return round(float(value), 4)
    return value


def metrics_row(label: str, metrics: dict, key_name: str = "Experiment") -> dict:
    return {key_name: label, **{key: _format_metric_value(value) for key, value in metrics.items()}}


def save_rows_csv(path: str | Path, rows: list[dict]):
    if not rows:
        return

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    headers = list(rows[0].keys())

    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved table to {path}")


def display_results_table(title: str, rows: list[dict]):
    if not rows:
        print(f"No rows provided for: {title}")
        return

    headers = list(rows[0].keys())
    lines = [f"### {title}", "", "| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(row.get(header, "")) for header in headers) + " |")
    display(Markdown("\n".join(lines)))


def run_detector_experiment(pretrained: bool, num_epochs: int = 10, lr: float = 1e-4):
    exp_model = build_model(TASK, pretrained=pretrained).to(DEVICE)
    exp_optimizer = torch.optim.Adam(exp_model.parameters(), lr=lr)
    exp_history = []

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(exp_model, train_loader, exp_optimizer, DEVICE)
        val_metrics = evaluate_val(exp_model, val_loader, DEVICE)
        exp_history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

    test_results = evaluate_model_on_loader(exp_model, test_loader, DEVICE)
    return exp_model, exp_history, test_results


@dataclass
class DetectorExperimentSpec:
    name: str
    train_generator_config: ShapeGenerationConfig
    eval_generator_config: ShapeGenerationConfig
    train_size: int | None = None
    use_augmentation: bool = False


def build_experiment_spec(experiment_group: str, variant: str) -> DetectorExperimentSpec:
    if experiment_group == "opacity":
        if variant == "high":
            cfg = ShapeGenerationConfig(opacity_range=(190, 255))
        elif variant == "low":
            cfg = ShapeGenerationConfig(opacity_range=(48, 128))
        else:
            raise ValueError(f"Unknown opacity variant: {variant}")
        return DetectorExperimentSpec(f"opacity/{variant}", cfg, cfg)

    if experiment_group == "difficulty":
        if variant == "easy":
            cfg = ShapeGenerationConfig(opacity_range=(180, 255), blur_range=(0.0, 0.4), noise_std_range=(0.0, 2.0), color_jitter=60, min_size_ratio=1/10, max_size_ratio=1/3)
        elif variant == "hard":
            cfg = ShapeGenerationConfig(opacity_range=(64, 144), blur_range=(0.8, 2.4), noise_std_range=(5.0, 12.0), color_jitter=18, min_size_ratio=1/16, max_size_ratio=1/5)
        else:
            raise ValueError(f"Unknown difficulty variant: {variant}")
        return DetectorExperimentSpec(f"difficulty/{variant}", cfg, cfg)

    if experiment_group == "train_size":
        cfg = DEFAULT_SHAPE_CONFIG
        if variant == "small":
            return DetectorExperimentSpec("train_size/small", cfg, cfg, train_size=1000)
        if variant == "large":
            return DetectorExperimentSpec("train_size/large", cfg, cfg, train_size=len(train_ids))
        raise ValueError(f"Unknown train_size variant: {variant}")

    if experiment_group == "augmentation":
        cfg = DEFAULT_SHAPE_CONFIG
        if variant == "with_aug":
            return DetectorExperimentSpec("augmentation/with_aug", cfg, cfg, use_augmentation=True)
        if variant == "without_aug":
            return DetectorExperimentSpec("augmentation/without_aug", cfg, cfg, use_augmentation=False)
        raise ValueError(f"Unknown augmentation variant: {variant}")

    raise ValueError(f"Unknown experiment group: {experiment_group}")


def build_loaders_for_spec(spec: DetectorExperimentSpec, batch_size: int = 4):
    selected_train_ids = train_ids if spec.train_size is None else train_ids[:spec.train_size]

    exp_train_ds = SyntheticShapeDataset(
        train_base,
        selected_train_ids,
        "train",
        TASK,
        get_transforms("train", use_augmentation=spec.use_augmentation),
        generator_config=spec.train_generator_config,
    )
    exp_val_ds = SyntheticShapeDataset(
        val_base,
        val_ids,
        "val",
        TASK,
        get_transforms("val"),
        generator_config=spec.eval_generator_config,
    )
    exp_test_ds = SyntheticShapeDataset(
        val_base,
        test_ids,
        "test",
        TASK,
        get_transforms("val"),
        generator_config=spec.eval_generator_config,
    )

    exp_train_loader = DataLoader(exp_train_ds, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)
    exp_val_loader = DataLoader(exp_val_ds, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate_fn)
    exp_test_loader = DataLoader(exp_test_ds, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate_fn)
    return exp_train_loader, exp_val_loader, exp_test_loader


def run_named_experiment(experiment_group: str, variant: str, pretrained: bool = True, num_epochs: int = 10, lr: float = 1e-4):
    spec = build_experiment_spec(experiment_group, variant)
    exp_train_loader, exp_val_loader, exp_test_loader = build_loaders_for_spec(spec)
    exp_model = build_model(TASK, pretrained=pretrained).to(DEVICE)
    exp_optimizer = torch.optim.Adam(exp_model.parameters(), lr=lr)
    exp_history = []

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(exp_model, exp_train_loader, exp_optimizer, DEVICE)
        val_metrics = evaluate_val(exp_model, exp_val_loader, DEVICE)
        exp_history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

    exp_test_results = evaluate_model_on_loader(exp_model, exp_test_loader, DEVICE)
    return spec, exp_model, exp_history, exp_test_results


In [ ]:
# Required comparison table — main model vs simple baseline
required_comparison_rows = [
    metrics_row("Faster R-CNN (fine-tuned)", test_metrics, key_name="Model"),
    metrics_row("Edge + Connected Components baseline", baseline_test_metrics, key_name="Model"),
]

display_results_table("Required Comparison — Main Model vs Baseline", required_comparison_rows)
save_rows_csv("results/required_comparison.csv", required_comparison_rows)

### Report Table Template — Main Model vs Baseline

| Model | precision@0.5 | recall@0.5 | f1@0.5 | mean_iou_matched |
| --- | --- | --- | --- | --- |
| Faster R-CNN (fine-tuned) |  |  |  |  |
| Edge + Connected Components baseline |  |  |  |  |


In [ ]:
# Extra comparison table — fine-tuned Faster R-CNN vs scratch Faster R-CNN
# Example usage:
# _, fine_tuned_history, fine_tuned_results = run_detector_experiment(pretrained=True,  num_epochs=10)
# _, scratch_history,    scratch_results    = run_detector_experiment(pretrained=False, num_epochs=10)

extra_comparison_rows = []
# Uncomment and use after running both experiments:
# extra_comparison_rows = [
#     metrics_row("Faster R-CNN (fine-tuned)", fine_tuned_results["metrics"], key_name="Model"),
#     metrics_row("Faster R-CNN (scratch)", scratch_results["metrics"], key_name="Model"),
# ]

if extra_comparison_rows:
    display_results_table("Extra Comparison — Fine-Tuned vs Scratch Faster R-CNN", extra_comparison_rows)
    save_rows_csv("results/extra_pretraining_comparison.csv", extra_comparison_rows)
else:
    print("Run both pretrained=True and pretrained=False experiments, then fill extra_comparison_rows.")

### Report Table Template — Fine-Tuned vs Scratch

| Model | precision@0.5 | recall@0.5 | f1@0.5 | mean_iou_matched |
| --- | --- | --- | --- | --- |
| Faster R-CNN (fine-tuned) |  |  |  |  |
| Faster R-CNN (scratch) |  |  |  |  |


In [ ]:
# Experiment 1 — high opacity vs low opacity
# Experiment 2 — easy shapes vs hard shapes
# These experiments change synthetic data generation for both train and evaluation.

# Example usage:
# opacity_runs = {}
# for variant in ["high", "low"]:
#     spec, _, history, results = run_named_experiment("opacity", variant, pretrained=True, num_epochs=5)
#     opacity_runs[variant] = {"spec": spec, "history": history, "results": results}
#
# opacity_rows = [
#     metrics_row("opacity/high", opacity_runs["high"]["results"]["metrics"]),
#     metrics_row("opacity/low", opacity_runs["low"]["results"]["metrics"]),
# ]
# display_results_table("Experiment — High Opacity vs Low Opacity", opacity_rows)
# save_rows_csv("results/opacity_metrics.csv", opacity_rows)
#
# difficulty_runs = {}
# for variant in ["easy", "hard"]:
#     spec, _, history, results = run_named_experiment("difficulty", variant, pretrained=True, num_epochs=5)
#     difficulty_runs[variant] = {"spec": spec, "history": history, "results": results}
#
# difficulty_rows = [
#     metrics_row("difficulty/easy", difficulty_runs["easy"]["results"]["metrics"]),
#     metrics_row("difficulty/hard", difficulty_runs["hard"]["results"]["metrics"]),
# ]
# display_results_table("Experiment — Easy Shapes vs Hard Shapes", difficulty_rows)
# save_rows_csv("results/difficulty_metrics.csv", difficulty_rows)

print("Experiment templates ready: opacity/high-vs-low and difficulty/easy-vs-hard.")

### Report Table Template — Opacity and Difficulty Experiments

| Experiment | precision@0.5 | recall@0.5 | f1@0.5 | mean_iou_matched | Short interpretation |
| --- | --- | --- | --- | --- | --- |
| opacity/high |  |  |  |  |  |
| opacity/low |  |  |  |  |  |
| difficulty/easy |  |  |  |  |  |
| difficulty/hard |  |  |  |  |  |


In [ ]:
# Experiment 3 — small training set vs larger training set
# Experiment 4 — with data augmentation vs without data augmentation
# These experiments keep evaluation fixed and change the training setup.

# Example usage:
# train_size_runs = {}
# for variant in ["small", "large"]:
#     spec, _, history, results = run_named_experiment("train_size", variant, pretrained=True, num_epochs=5)
#     train_size_runs[variant] = {"spec": spec, "history": history, "results": results}
#
# train_size_rows = [
#     metrics_row("train_size/small", train_size_runs["small"]["results"]["metrics"]),
#     metrics_row("train_size/large", train_size_runs["large"]["results"]["metrics"]),
# ]
# display_results_table("Experiment — Small Training Set vs Larger Training Set", train_size_rows)
# save_rows_csv("results/train_size_metrics.csv", train_size_rows)
#
# augmentation_runs = {}
# for variant in ["without_aug", "with_aug"]:
#     spec, _, history, results = run_named_experiment("augmentation", variant, pretrained=True, num_epochs=5)
#     augmentation_runs[variant] = {"spec": spec, "history": history, "results": results}
#
# augmentation_rows = [
#     metrics_row("augmentation/without_aug", augmentation_runs["without_aug"]["results"]["metrics"]),
#     metrics_row("augmentation/with_aug", augmentation_runs["with_aug"]["results"]["metrics"]),
# ]
# display_results_table("Experiment — With Data Augmentation vs Without Data Augmentation", augmentation_rows)
# save_rows_csv("results/augmentation_metrics.csv", augmentation_rows)

print("Experiment templates ready: train_size/small-vs-large and augmentation/with-vs-without.")

### Report Table Template — Train Size and Augmentation Experiments

| Experiment | precision@0.5 | recall@0.5 | f1@0.5 | mean_iou_matched | Short interpretation |
| --- | --- | --- | --- | --- | --- |
| train_size/small |  |  |  |  |  |
| train_size/large |  |  |  |  |  |
| augmentation/without_aug |  |  |  |  |  |
| augmentation/with_aug |  |  |  |  |  |


---
## 10. Prediction Visualizations

At least **12** test images required (§9).  
Must include: successful predictions, failure cases, positive images, negative images.

In [ ]:
def visualize_predictions(
    images,
    targets,
    predictions,
    n: int = 12,
    save_dir: str | Path = "figures/prediction_visualizations/general",
    score_threshold: float = 0.5,
):
    """
    Visualize at least 12 test images (§9).
    Include: successful predictions, failure cases, positive images, negative images.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    def filter_prediction(prediction):
        boxes = _to_box_tensor(prediction.get("boxes"))
        scores = prediction.get("scores")
        if scores is None:
            keep = torch.ones((len(boxes),), dtype=torch.bool)
        else:
            keep = scores.detach().cpu() >= score_threshold
        return {
            "boxes": boxes[keep],
            "scores": scores.detach().cpu()[keep] if scores is not None else None,
        }

    def categorize_case(target, filtered_prediction):
        gt_boxes = _to_box_tensor(target.get("boxes"))
        pred_boxes = _to_box_tensor(filtered_prediction.get("boxes"))
        matches = _match_boxes_greedy(pred_boxes, gt_boxes, iou_threshold=0.5)

        is_positive = bool(target.get("is_positive", False))
        if is_positive:
            if len(matches) == len(gt_boxes) and len(pred_boxes) == len(gt_boxes):
                return "success_positive"
            return "failure_positive"

        if len(pred_boxes) == 0:
            return "success_negative"
        return "failure_negative"

    filtered_predictions = [filter_prediction(prediction) for prediction in predictions]
    buckets = {
        "success_positive": [],
        "failure_positive": [],
        "success_negative": [],
        "failure_negative": [],
    }

    for idx, (target, prediction) in enumerate(zip(targets, filtered_predictions)):
        buckets[categorize_case(target, prediction)].append(idx)

    selection = []
    preferred_order = ["success_positive", "failure_positive", "success_negative", "failure_negative"]
    for bucket_name in preferred_order:
        selection.extend(buckets[bucket_name][: max(2, n // 4)])

    if len(selection) < n:
        for idx in range(len(images)):
            if idx not in selection:
                selection.append(idx)
            if len(selection) >= n:
                break

    selection = selection[:n]
    rows = int(np.ceil(n / 3))
    cols = 3
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    axes = np.array(axes).reshape(-1)

    for plot_idx, sample_idx in enumerate(selection):
        image = images[sample_idx]
        target = targets[sample_idx]
        prediction = filtered_predictions[sample_idx]
        category = categorize_case(target, prediction)

        image_np = image.detach().cpu().permute(1, 2, 0).numpy()
        image_np = np.clip(image_np, 0.0, 1.0)

        ax = axes[plot_idx]
        ax.imshow(image_np)

        for x1, y1, x2, y2 in _to_box_tensor(target.get("boxes")):
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))

        pred_boxes = _to_box_tensor(prediction.get("boxes"))
        pred_scores = prediction.get("scores")
        for box_idx, (x1, y1, x2, y2) in enumerate(pred_boxes):
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2, linestyle="--"))
            if pred_scores is not None:
                ax.text(x1, max(0, y1 - 4), f"{float(pred_scores[box_idx]):.2f}", color="red", fontsize=9, backgroundcolor="white")

        ax.set_title(f"{category} | gt={len(_to_box_tensor(target.get('boxes')))} | pred={len(pred_boxes)}")
        ax.axis("off")

        sample_path = save_dir / f"sample_{plot_idx + 1:02d}_{category}.png"
        sample_fig, sample_ax = plt.subplots(figsize=(6, 6))
        sample_ax.imshow(image_np)
        for x1, y1, x2, y2 in _to_box_tensor(target.get("boxes")):
            sample_ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
        for box_idx, (x1, y1, x2, y2) in enumerate(pred_boxes):
            sample_ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2, linestyle="--"))
            if pred_scores is not None:
                sample_ax.text(x1, max(0, y1 - 4), f"{float(pred_scores[box_idx]):.2f}", color="red", fontsize=9, backgroundcolor="white")
        sample_ax.set_title(f"{category} | gt={len(_to_box_tensor(target.get('boxes')))} | pred={len(pred_boxes)}")
        sample_ax.axis("off")
        sample_fig.savefig(sample_path, dpi=160, bbox_inches="tight")
        plt.close(sample_fig)

    for ax in axes[len(selection):]:
        ax.axis("off")

    fig.tight_layout()
    grid_path = save_dir / "prediction_grid.png"
    fig.savefig(grid_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved prediction grid to {grid_path}")


visualize_predictions(all_images, all_targets, all_predictions, n=12)

---
## 11. Failure Cases

Briefly discuss which types of synthetic additions are difficult for the model to detect and why.

---
## 12. Conclusion

Summarize findings, limitations, and possible improvements.